In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import calpred
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from typing import List
import statsmodels.api as sm
from scipy import stats
import pickle
from tqdm import tqdm
import calpgs
import copy
import warnings
import os
import pandas as pd
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)
import matplotlib as mpl

mpl.rcParams['font.family'] = "Nimbus Sans"
mpl.rcParams['lines.linewidth'] = 1.5
mpl.rcParams['axes.linewidth'] = 1.5
mpl.rcParams['font.size'] = 16

# sim pheno

In [2]:
# Assume we have obtained GV from simulation_wocontext.ipynb

In [ ]:

for h2 in [0.5]:
    for causal_percent in [1]:
        print(causal_percent)

        for gamma in range(21):
            j = gamma
            gamma = gamma * 0.05
            print(gamma)
            
            fam = pd.read_csv("geno.fam",header=None,sep="\t")
            np.random.seed(42)
            fam[2] = np.random.normal(0,1,size=len(fam))
            fam = fam[np.arange(3)]
            
            
            for i in tqdm(range(30)):
                gv = pd.read_csv(f"sim_gv.h2{h2}.causal{causal_percent}.seed{i+1}.sscore",sep="\t")
                fam[3] = gv.SCORE1_SUM
                fam[3] = (fam[3]-fam[3].mean())/fam[3].std()
            

                fam[4] = np.random.normal(
                    loc=fam[3],
                    scale=np.sqrt(np.exp(np.log(1/h2-1)+fam[2]*gamma)),
                ) 
                
                fam[4] = (fam[4]-fam[4].mean())/fam[4].std()
                df_trait = copy.copy(fam)
                df_trait.columns=["FID","IID","quant","GV","y"]

                df_trait.sample(frac=1,random_state=i).iloc[:60000].to_csv(f"sim_df.h2{h2}.causal{causal_percent}.gamma{j*5}.seed{i+1}.tsv",sep="\t",index=None)
            
            

# GWAS

In [ ]:

for h2 in [0.5]:
    for causal_percent in [1]:
    

        for gamma in range(21):
            for i in tqdm(range(30)):
                df = pd.read_csv(f"geno_sim_fig2/dataframe/sim_df.h2{h2}.causal{causal_percent}.gamma{gamma*5}.seed{i+1}.tsv",sep="\t")
                df = df.iloc[:50000]
                
                covar = df[["FID","IID","quant"]]
                pheno = df[["FID","IID","y"]]
                
                for j in range(5):
                    index = df.iloc[j*10000:(j+1)*10000].IID
                    
                    pdcovar = covar[~covar.IID.isin(index)]
                    pdpheno = pheno[~pheno.IID.isin(index)]
                    
                    pdpheno.to_csv(f"pheno.h2{h2}.causal{causal_percent}.gamma{gamma*5}.seed{i}.cv{j}.tsv",sep="\t",index=None)
                    pdcovar.to_csv(f"covar.h2{h2}.causal{causal_percent}.gamma{gamma*5}.seed{i}.cv{j}.tsv",sep="\t",index=None)
    
        
    

In [ ]:
for h2 in [0.5]:
    for causal_percent in [1]:
        for gamma in range(21):
            for i in tqdm(range(30)):
                for j in range(5):
                  !plink \
                    --bfile geno \
                    --pheno pheno.h2{h2}.causal{causal_percent}.gamma{gamma*5}.seed{i}.cv{j}.tsv \
                    --pheno-col-nums 3 \
                    --covar covar.h2{h2}.causal{causal_percent}.gamma{gamma*5}.seed{i}.cv{j}.tsv \
                    --covar-col-nums 3 \
                    --glm hide-covar \
                    --no-input-missing-phenotype \
                    --threads 8 \
                    --out gwas.h2{h2}.causal{causal_percent}.gamma{gamma*5}.seed{i}.cv{j}



# PGS

In [ ]:
for h2 in [0.5]:
    
    for causal_percent in [1]:
        print(causal_percent)
        for gamma in range(21):
            gamma = gamma * 5
            print(gamma)
            for seed in tqdm(range(30)):
                for cv in range(5):
                    sst = pd.read_csv(f"gwas.h2{h2}.causal{causal_percent}.gamma{gamma}.seed{seed}.cv{cv}.y.glm.linear",sep="\t").drop(columns=["A1"])
                    
                    sst.rename(columns={"ID":"SNP","ALT":"A1","REF":"A2"},inplace=True)
                    
                    sst[["SNP","A1","A2","BETA","SE"]].to_csv(f"gwas.h2{h2}.causal{causal_percent}.gamma{gamma}.seed{seed}.cv{cv}.sst",sep="\t",index=None)
                
    

In [ ]:
for h2 in [0.5]:
    
    for causal_percent in [1]:
        print(causal_percent)
        for gamma in range(21):
            gamma = gamma * 5
            print(gamma)
            for seed in tqdm(range(30)):
                for cv in range(5):
                  !python PRScs.py \
                    --ref_dir=ldblk_1kg_eur \
                    --bim_prefix=geno \
                    --sst_file=gwas.h2{h2}.causal{causal_percent}.gamma{gamma}.seed{seed}.cv{cv}.sst \
                    --n_gwas=40000 \
                    --chrom=1 \
                    --out_dir=pgs.h2{h2}.causal{causal_percent}.gamma{gamma}.seed{seed}.cv{cv}



In [ ]:
for h2 in [0.5]:
    
    for causal_percent in [1]:
        print(causal_percent)
        for gamma in range(21):
            gamma = gamma * 5
            print(gamma)
            for seed in tqdm(range(30)):
                for cv in range(5):
                  !plink \
                    --bfile geno \
                    --score pgs.h2{h2}.causal{causal_percent}.gamma{gamma}.seed{seed}.cv{cv}_pst_eff_a1_b0.5_phiauto_chr1.txt 2 4 6 cols=+scoresums \
                    --threads 8 \
                    --out score.h2{h2}.causal{causal_percent}.gamma{gamma}.seed{seed}.cv{cv}



In [ ]:
for h2 in [0.5]:
    for causal_percent in [1]:
        for gamma in range(21):
            gamma = gamma *5
            print(gamma)
            for i in tqdm(range(30)):
                df = pd.read_csv(f"sim_df.h2{h2}.causal{causal_percent}.gamma{gamma}.seed{i+1}.tsv",sep="\t")
                for cv in range(5):
                    
                    pgs = pd.read_csv(f"score.h2{h2}.causal{causal_percent}.gamma{gamma}.seed{i}.cv{cv}.sscore",sep="\t")
                    df = df.merge(pgs[["IID","SCORE1_SUM"]].rename(columns={"SCORE1_SUM":f"PGS_{cv}"}),on="IID",how="left")
        
                df.to_csv(f"sim_df_PGS.h2{h2}.causal{causal_percent}.gamma{gamma}.seed{i}.tsv",sep="\t",index=None)
            

# Evaluate CalPred + Generic

In [ ]:
def evaluate(N_SIM,gamma,h2,causal,decile,alpha):


    dict_stats_sum = dict()

    for adjust in ["all"]:
        if adjust == "all":
            adjust_cols = ["quant","quant**2"]
        elif adjust == "none":
            adjust_cols = []


        dict_df_coverage = dict()
        dict_df_r2 = dict()
        dict_df_length = dict()

        df_coverage = []
        df_r2 = []
        df_length = []
        for seed in tqdm(range(N_SIM)):

            df = pd.read_csv(f"sim_df_PGS.h2{h2}.causal{causal}.gamma{gamma}.seed{seed}.tsv",sep="\t")
            df["quant**2"] = df["quant"]**2
            if decile:
                col =  "quant"
                col_q = pd.qcut(df[col], q=10).cat.codes
                df[f"{col}_q"] = col_q
            
            df_cal = df.iloc[:10000]
            df_test = df.iloc[50000:]


            
            model = calpred.fit(y=df_cal["y"],x=df_cal["PGS_0"],z=df_cal[adjust_cols])
            mean,std = calpred.predict(x=df_test["PGS_0"],z=df_test[adjust_cols],model_fit=model)
            df_test["mean"] = mean
            df_test["std"] = std
            df_test["lower"] = mean-stats.norm.ppf(0.5 + alpha/2)*std
            df_test["upper"] = mean+stats.norm.ppf(0.5 + alpha/2)*std

            #df_test[["IID","GV","y","mean","lower","upper","std","AGE_q","PC1_q","SEX_q"]].to_csv(f"geno_sim/output_cache/{adjust}.seed{seed}.pred.tsv",sep="\t",index=None)

            tmp_cov = {}
            tmp_r2 = {}
            tmp_length = {}
            for group_col in [None, "quant_q"]:
                if group_col == None:
                    tmp_cov["marginal"]=df_test["y"].between(df_test.lower,df_test.upper).sum()/len(df_test)
                    tmp_r2["marginal"]=stats.pearsonr(df_test["PGS_0"],df_test.y).statistic**2
                    tmp_length["marginal"] = df_test["std"].mean()
                    continue
                for v in df_test[group_col].unique():
                    current = df_test[df_test[group_col]==v]
                    tmp_cov[f"{group_col}_{v}"]=current["y"].between(current.lower,current.upper).sum()/len(current)
                    tmp_r2[f"{group_col}_{v}"]=stats.pearsonr(current["PGS_0"],current.y).statistic**2
                    tmp_length[f"{group_col}_{v}"] = current["std"].mean()
            tmp_cov = pd.Series(tmp_cov)
            tmp_r2= pd.Series(tmp_r2)
            tmp_length = pd.Series(tmp_length)
            
            df_coverage.append(tmp_cov)
            df_r2.append(tmp_r2)
            df_length.append(tmp_length)
        n_calibrate = 10000
        dict_df_coverage[n_calibrate] = pd.concat(df_coverage, axis=1).T
        dict_df_r2[n_calibrate] = pd.concat(df_r2, axis=1).T
        dict_df_length[n_calibrate] = pd.concat(df_length, axis=1).T

        df_stats = {
            "n": [],
            "seed": [],
            "col": [],
            "coverage": [],
            "r2": [],
            "length": [],
        }

        for n in dict_df_coverage:
            for col in dict_df_coverage[n].columns:
                covs = dict_df_coverage[n][col].values
                df_stats["n"].extend([n] * len(covs))
                df_stats["seed"].extend(np.arange(len(covs)))
                df_stats["col"].extend([col] * len(covs))
                df_stats["coverage"].extend(covs)
                df_stats["r2"].extend(dict_df_r2[n][col])
                df_stats["length"].extend(dict_df_length[n][col])

        #
        dict_stats_sum[adjust] = pd.DataFrame(df_stats)

    # format into long table
    df_stats = []
    for adjust in dict_stats_sum:
        df_tmp = dict_stats_sum[adjust]
        df_tmp.insert(0, "adjust", adjust)
        df_stats.append(df_tmp)
    df_stats = pd.concat(df_stats)


    df_stats.to_csv(f"calPred.gamma{gamma}.h2{h2}.causal{causal}.alpha{int(alpha*100)}.real.decile.stats.tsv", sep="\t", index=False)

h2=0.5
causal=1

for gamma in (range(0,21)):
    evaluate(30,gamma*5,h2,causal,decile=True,alpha=0.95)
    evaluate(30,gamma*5,h2,causal,decile=True,alpha=0.5)

# Evaluate PredInterval

In [ ]:
def PredInterval_fit_pred(df_train,df_test,alpha):

    ricv = []

    print("start building RiCV")


    for i in range(5):
    

        current = df_train.iloc[i*10000:(i+1)*10000]
        ricv.append(abs(current.y-current[f"PGS_{i}"]).values)  

    
    upper = []
    lower = []
    means = []

    print("start calculating test residuals")

    
    
    for i in range(5):
        resid = ricv[i]
        
        mean = df_test[f"PGS_{i}"]
        means.append(mean)
        for r in resid:
            upper.append(mean+r)
            lower.append(mean-r)
    
    upper = pd.concat(upper,axis=1)
    upper = np.quantile(upper,alpha,axis=1)
    lower = pd.concat(lower,axis=1)
    lower = np.quantile(lower,1-alpha,axis=1)
    means = pd.concat(means,axis=1).mean(axis=1)

    std = (np.array(upper)-np.array(lower))/stats.norm.ppf(0.5 + alpha/2)/2
    

    return means, std, upper ,lower

def evaluate_PredInterval_main(df_train,df_test,alpha):


    print("start fitting predint")
    mean,std,upper,lower = PredInterval_fit_pred(df_train,df_test,alpha)
    print("Done")
    df_test["upper"] = upper
    df_test["lower"] = lower
    df_test["std"] = std
    df_test["mean"] = mean


    tmp_cov = {}
    tmp_r2 = {}
    tmp_length = {}
    for group_col in [None, "quant_q"]:
        if group_col == None:
            tmp_cov["marginal"]=df_test["y"].between(df_test.lower,df_test.upper).sum()/len(df_test)
            tmp_r2["marginal"]=stats.pearsonr(df_test["mean"],df_test.y).statistic**2
            tmp_length["marginal"] = df_test["std"].mean()
            continue
        for v in df_test[group_col].unique():
            current = df_test[df_test[group_col]==v]
            tmp_cov[f"{group_col}_{v}"]=current["y"].between(current.lower,current.upper).sum()/len(current)
            tmp_r2[f"{group_col}_{v}"]=stats.pearsonr(current["mean"],current.y).statistic**2
            tmp_length[f"{group_col}_{v}"] = current["std"].mean()
    tmp_cov = pd.Series(tmp_cov)
    tmp_r2= pd.Series(tmp_r2)
    tmp_length = pd.Series(tmp_length)
    
    return (
        pd.Series(tmp_cov),
        pd.Series(tmp_r2),
        pd.Series(tmp_length),
    )


def evaluate_PredInterval_wrapper(sim,scheme="PredInterval",gamma=None,h2=None,causal=None,decile=True,alpha=0.95):
    

    
    dict_stats_sum = dict()
    
    dict_df_coverage = dict()
    dict_df_r2 = dict()
    dict_df_length = dict()
    
    df_coverage = []
    df_r2 = []
    df_length = []
    seed = int(sim)-1
    df = pd.read_csv(f"sim_df_PGS.h2{h2}.causal{causal}.gamma{gamma}.seed{seed}.tsv",sep="\t")
    if decile:
        col =  "quant"
        col_q = pd.qcut(df[col], q=10).cat.codes
        df[f"{col}_q"] = col_q
    df_train = df.iloc[:50000] 
    df_test = df.iloc[50000:]

    
    tmp_cov, tmp_r2, tmp_length = evaluate_PredInterval_main(
        df_train, df_test,alpha
    )
    df_coverage.append(tmp_cov)
    df_r2.append(tmp_r2)
    df_length.append(tmp_length)
    n_calibrate = 10000
    dict_df_coverage[n_calibrate] = pd.concat(df_coverage, axis=1).T
    dict_df_r2[n_calibrate] = pd.concat(df_r2, axis=1).T
    dict_df_length[n_calibrate] = pd.concat(df_length, axis=1).T
    df_stats = {
        "n": [],
        "seed": [],
        "col": [],
        "coverage": [],
        "r2": [],
        "length": [],
    }
    
    # summarize coverage / R2
    for n in dict_df_coverage:
        for col in dict_df_coverage[n].columns:
            covs = dict_df_coverage[n][col].values
            df_stats["n"].extend([n] * len(covs))
            df_stats["seed"].extend(np.arange(len(covs)))
            df_stats["col"].extend([col] * len(covs))
            df_stats["coverage"].extend(covs)
            df_stats["r2"].extend(dict_df_r2[n][col])
            df_stats["length"].extend(dict_df_length[n][col])
    
    
    
    dict_stats_sum[scheme] = pd.DataFrame(df_stats)
    
    # format into long table
    df_stats = []
    for adjust in dict_stats_sum:
        df_tmp = dict_stats_sum[adjust]
        df_tmp.insert(0, "adjust", adjust)
        df_stats.append(df_tmp)
    df_stats = pd.concat(df_stats)


    
    df_stats.to_csv(f"{scheme}.{seed}.gamma{gamma}.h2{h2}.causal{causal}.alpha{int(alpha*100)}.decile.stats.tsv", sep="\t", index=False)





h2=0.5
causal=1

for sim in range(1,31):
    for scheme in ["PredInterval"]:
        for gamma in range(0,21):
            evaluate_PredInterval_wrapper(sim,scheme,gamma=gamma*5,h2=h2,causal=causal,alpha=0.95)


In [ ]:
h2=0.5
causal=1

for gamma in tqdm(range(0,21)):
    #for pgsuse in ["cv","single"]:
    gamma = gamma * 5
    for pgsuse in ["cv"]:
        for scheme in ["PredInterval"]:
            df = []
            for i in range(30):

                df.append(pd.read_csv(f"{scheme}.{i}.gamma{gamma}.h2{h2}.causal{causal}.alpha95.decile.stats.tsv",sep="\t"))
            pd.concat(df).to_csv(f"{scheme}.gamma{gamma}.h2{h2}.causal{causal}.alpha95.decile.stats.tsv",sep="\t",index=None)
        
